# Opyna - Inteligencia de Voz del Cliente

Pipeline reproducible del TFM: de resenas en texto libre a inteligencia de negocio accionable.

**Dataset:** Amazon Reviews 2023 (McAuley Lab), categoria Electronics, 200.000 resenas balanceadas.

**Fases:** EDA -> baseline clasico -> ensemble -> transformers (sentimiento y rating) ->
ABSA por aspecto -> validacion -> temas emergentes -> interpretabilidad ->
capa generativa -> caso de negocio.

Los modelos entrenados se publican en Hugging Face; el dataset se regenera desde el origen.

> Requiere GPU. Con A100 el pipeline completo tarda ~40 min.

## 0. Setup

In [ ]:
# --- Setup reproducible: no depende de Google Drive ---
import os, datetime

# Carpeta de salida configurable (por defecto, local al entorno).
# Para persistir en Colab: os.environ["OPYNA_OUT"] = "<tu carpeta>"
OUT = os.environ.get("OPYNA_OUT", "./artefactos")
for sub in ["", "/modelos", "/resultados", "/figuras"]:
    os.makedirs(OUT + sub, exist_ok=True)

MEM = f"{OUT}/memoria_resultados.md"
def mem(text, mode="a"):
    with open(MEM, mode, encoding="utf-8") as f:
        f.write(text + "\n")

# Versiones fijadas: transformers 5 elimino pipeline('translation')
# y cambio la API de Trainer. Este pipeline requiere la serie 4.
!pip -q install "transformers<5" "huggingface-hub<1.0" sentencepiece "datasets==2.19.0" \
    "fsspec<=2024.3.1" pydantic-settings pyarrow sacremoses

import torch
print("GPU:", torch.cuda.is_available(),
      "->", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Salida:", os.path.abspath(OUT))

In [ ]:
# --- Datos: Amazon Reviews 2023 (Electronics), 200.000 resenas balanceadas ---
# 40.000 por cada valoracion (1-5 estrellas), semilla fija.
#
# El modulo de descarga forma parte de este repositorio, en src/opyna/.
# Ajusta CODIGO si mueves el notebook de sitio.

import sys, os

CODIGO = os.environ.get("OPYNA_CODIGO", "../src")
if CODIGO not in sys.path:
    sys.path.insert(0, CODIGO)

from opyna.data.download import download_sample

import pandas as pd
df = download_sample(category="Electronics", n_per_rating=40000)
print("resenas:", len(df), "| columnas:", list(df.columns))

# El pipeline lee de data/processed/ (ruta relativa, como el modulo original)
os.makedirs("data/processed", exist_ok=True)
df.to_parquet("data/processed/reviews_electronics.parquet", index=False)

# copia en la carpeta de salida, para conservarla junto al resto de artefactos
os.makedirs(OUT + "/resultados", exist_ok=True)
df.to_parquet(OUT + "/resultados/reviews_electronics_200k.parquet", index=False)
print("guardado en data/processed/ y en", OUT + "/resultados/")


## 1. EDA y baseline clasico (TF-IDF + Regresion Logistica)

In [ ]:
# PASO 1: EDA + Baseline (TF-IDF + LogReg) -> resultados + memoria a Drive
import pandas as pd, numpy as np, json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
df = pd.read_parquet("data/processed/reviews_electronics.parquet")
df["texto"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()

# --- EDA ---
n = len(df); nulos = int(df[["rating","title","text"]].isna().sum().sum())
verif = round(100*df["verified_purchase"].mean(),1)
df["n_palabras"] = df["texto"].str.split().str.len()
len_med = round(df["n_palabras"].median(),1); len_mean = round(df["n_palabras"].mean(),1)
help_mean = round(df["helpful_vote"].mean(),2); help_pct = round(100*(df["helpful_vote"]>=1).mean(),1)
len_est = {int(k):round(v,1) for k,v in df.groupby("rating")["n_palabras"].mean().items()}
eda = {"n":n,"nulos":nulos,"verificada_pct":verif,"long_mediana":len_med,"long_media":len_mean,"helpful_media":help_mean,"helpful_pct_ge1":help_pct,"long_por_estrella":len_est}
json.dump(eda, open(f"{OUT}/resultados/eda_200k.json","w"), indent=2)
print("EDA:", eda)
mem("## 1. EDA\n"
    f"- Total {n:,} resenas, **{nulos} nulos** en columnas clave.\n"
    f"- Compra verificada: **{verif}%**.\n"
    f"- Longitud: mediana {len_med} palabras / media {len_mean}.\n"
    f"- Helpful votes: media {help_mean}, {help_pct}% con >=1 voto.\n"
    f"- Longitud media por estrella: {len_est}.\n"
    "- Guardado: `resultados/eda_200k.json`\n")

# --- Baseline ---
def a3(r): return "negativo" if r<=2 else ("neutral" if r==3 else "positivo")
df["sent3"] = df["rating"].apply(a3)
Xtr,Xte,ytr,yte = train_test_split(df["texto"], df["sent3"], test_size=0.2, random_state=42, stratify=df["sent3"])
vec = TfidfVectorizer(ngram_range=(1,2), min_df=5, max_features=100000, sublinear_tf=True)
Xtr_v = vec.fit_transform(Xtr); Xte_v = vec.transform(Xte)
clf = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1).fit(Xtr_v, ytr)
yp = clf.predict(Xte_v)
acc = accuracy_score(yte,yp); f1m = f1_score(yte,yp,average="macro")
rep = classification_report(yte,yp, digits=3)
cm = confusion_matrix(yte,yp, labels=["negativo","neutral","positivo"])
print(f"=== Baseline 200k: accuracy={acc:.3f}  f1_macro={f1m:.3f} ===")
print(rep); print("Matriz confusion (neg/neu/pos):\n", cm)
with open(f"{OUT}/resultados/baseline_200k.txt","w") as f:
    f.write(f"Baseline TF-IDF+LogReg 200k\naccuracy={acc:.4f} f1_macro={f1m:.4f}\n\n{rep}\n\nMatriz (neg/neu/pos):\n{cm}\n")
mem("## 2. Baseline (TF-IDF ngram(1,2) min_df=5 + Regresion Logistica balanceada, split 80/20)\n"
    f"- **Accuracy {acc:.3f} / F1-macro {f1m:.3f}**.\n"
    f"```\n{rep}\n```\n"
    "- Guardado: `resultados/baseline_200k.txt`\n")
print("PASO 1 OK -> Drive")

## 2. Baseline ensemble (LightGBM)

In [ ]:
# PASO 9 - Baseline ensemble (TF-IDF 30k + LightGBM) -> completa 5.1 (lineales y ensemble) -> Drive + memoria
import os, glob, time, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
try:
    import lightgbm as lgb
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','lightgbm'],check=True); import lightgbm as lgb
BASE=OUT; SEED=42
cand=[c for c in glob.glob(BASE+'/**/*.parquet', recursive=True) if 'reviews' in c.lower()]
df=pd.read_parquet(cand[0]); TXT='text' if 'text' in df.columns else 'texto'
df['texto']=(df['title'].fillna('')+' '+df[TXT].fillna('')).str.strip()
df['y']=df['rating'].apply(lambda r: 0 if r<=2 else (1 if r==3 else 2))
Xtr,Xte,ytr,yte=train_test_split(df['texto'], df['y'], test_size=0.2, random_state=SEED, stratify=df['y'])
vec=TfidfVectorizer(ngram_range=(1,2), min_df=5, max_features=30000, sublinear_tf=True)
Xtr_v=vec.fit_transform(Xtr); Xte_v=vec.transform(Xte)
print('vectorizado:', Xtr_v.shape)
t0=time.time()
clf=lgb.LGBMClassifier(objective='multiclass', num_class=3, n_estimators=200, learning_rate=0.1, num_leaves=31, force_row_wise=True, verbosity=-1, n_jobs=-1, random_state=SEED)
clf.fit(Xtr_v, ytr); mins=(time.time()-t0)/60
yp=clf.predict(Xte_v)
acc=accuracy_score(yte,yp); f1m=f1_score(yte,yp,average='macro')
rep=classification_report(yte,yp,target_names=['negativo','neutral','positivo'],digits=3)
cm=confusion_matrix(yte,yp)
print('ENSEMBLE LightGBM: acc=%.3f f1m=%.3f (%.1f min)'%(acc,f1m,mins))
print(rep); print(cm)
open(BASE+'/resultados/baseline_ensemble_200k.txt','w').write('LightGBM TF-IDF(30k) acc=%.4f f1m=%.4f%sReport:%s%s'%(acc,f1m,chr(10),chr(10),rep))
L=['','## 10. Baseline ensemble (TF-IDF 30k + LightGBM) - completa 5.1 (lineales y ensemble)','- Mismo split 80/20 (seed 42), 3 clases neg/neu/pos. TF-IDF reducido a 30k features por tratabilidad del boosting.','- LightGBM (200 arboles): acc %.3f / F1-macro %.3f (entreno %.1f min).'%(acc,f1m,mins),'- Comparativa 3-clases (acc/F1m): LogReg 0.767/0.734 | LightGBM %.3f/%.3f | DistilBERT 0.818/0.768.'%(acc,f1m),'- Guardado: resultados/baseline_ensemble_200k.txt']
open(BASE+'/memoria_resultados_200k.md','a').write(chr(10).join(L)+chr(10))
print('OK Paso9 ensemble -> Drive')

## 3. Sentimiento y rating con DistilBERT (fine-tuning)

In [ ]:
# PASO 2: Fine-tuning DistilBERT -> SENTIMIENTO (3 clases) + RATING (5 estrellas). Modelos + metricas a Drive
import pandas as pd, numpy as np, torch, time
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
df = pd.read_parquet("data/processed/reviews_electronics.parquet")
df["texto"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
tk = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tok_fn(b): return tk(b["texto"], truncation=True, max_length=256)

def entrena(col, nombre, num_labels):
    tr, te = train_test_split(df, test_size=0.2, random_state=42, stratify=df[col])
    dtr = Dataset.from_pandas(tr[["texto",col]].rename(columns={col:"label"}).reset_index(drop=True)).map(tok_fn, batched=True)
    dte = Dataset.from_pandas(te[["texto",col]].rename(columns={col:"label"}).reset_index(drop=True)).map(tok_fn, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)
    args = TrainingArguments(output_dir=f"/content/out_{nombre}", num_train_epochs=2,
        per_device_train_batch_size=64, per_device_eval_batch_size=128, learning_rate=2e-5,
        weight_decay=0.01, fp16=True, seed=42, logging_steps=200, report_to="none", save_strategy="no")
    trn = Trainer(model=model, args=args, train_dataset=dtr, data_collator=DataCollatorWithPadding(tk))
    t0=time.time(); trn.train(); mins=(time.time()-t0)/60
    yp = trn.predict(dte).predictions.argmax(-1); yt = np.array(te[col])
    return model, yt, yp, mins

# --- SENTIMIENTO (neg=0/neu=1/pos=2) ---
df["sent3"] = df["rating"].apply(lambda r: 0 if r<=2 else (1 if r==3 else 2))
m_s, yt_s, yp_s, min_s = entrena("sent3", "sent", 3)
acc_s=accuracy_score(yt_s,yp_s); f1_s=f1_score(yt_s,yp_s,average="macro")
rep_s=classification_report(yt_s,yp_s,target_names=["negativo","neutral","positivo"],digits=3)
m_s.save_pretrained(f"{OUT}/modelos/sentimiento_200k"); tk.save_pretrained(f"{OUT}/modelos/sentimiento_200k")
open(f"{OUT}/resultados/sentimiento_200k.txt","w").write(f"acc={acc_s:.4f} f1m={f1_s:.4f} ({min_s:.1f}min)\n\n{rep_s}")
print(f"SENTIMIENTO: acc={acc_s:.3f} f1m={f1_s:.3f} ({min_s:.1f}min)\n{rep_s}")
mem("## 3. Sentimiento (DistilBERT fine-tuned, 3 clases, 2 epocas, batch 64)\n"
    f"- **Accuracy {acc_s:.3f} / F1-macro {f1_s:.3f}** (entreno {min_s:.1f} min en A100).\n"
    f"```\n{rep_s}\n```\n- Modelo guardado en Drive: `modelos/sentimiento_200k/`\n")

# --- RATING (0..4 = 1..5 estrellas) ---
df["rlabel"] = (df["rating"]-1).astype(int)
m_r, yt_r, yp_r, min_r = entrena("rlabel", "rating", 5)
acc_r=accuracy_score(yt_r,yp_r); f1_r=f1_score(yt_r,yp_r,average="macro"); acc_pm1=float(np.mean(np.abs(yt_r-yp_r)<=1))
rep_r=classification_report(yt_r,yp_r,target_names=["1*","2*","3*","4*","5*"],digits=3)
m_r.save_pretrained(f"{OUT}/modelos/rating_200k"); tk.save_pretrained(f"{OUT}/modelos/rating_200k")
open(f"{OUT}/resultados/rating_200k.txt","w").write(f"acc={acc_r:.4f} f1m={f1_r:.4f} acc_pm1={acc_pm1:.4f} ({min_r:.1f}min)\n\n{rep_r}")
print(f"RATING: acc_exacta={acc_r:.3f} f1m={f1_r:.3f} acc_pm1={acc_pm1:.3f} ({min_r:.1f}min)\n{rep_r}")
mem("## 4. Rating 5 estrellas (DistilBERT fine-tuned, 2 epocas, batch 64)\n"
    f"- **Accuracy exacta {acc_r:.3f} / F1-macro {f1_r:.3f} / Accuracy +-1 estrella {acc_pm1:.3f}**.\n"
    f"```\n{rep_r}\n```\n- Modelo guardado en Drive: `modelos/rating_200k/`\n")
print("PASO 2 OK -> Drive")

## 4. ABSA: sentimiento por aspecto (DeBERTa)

In [ ]:
# PASO 3: ABSA (DeBERTa) + agregacion por aspecto -> parquet + tabla + grafico a Drive
import pandas as pd, numpy as np, re, time, os, matplotlib.pyplot as plt
from transformers import pipeline
df = pd.read_parquet("data/processed/reviews_electronics.parquet").reset_index(drop=True)
df["texto"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
low = df["texto"].str.lower()
ASPECTOS = {
  "bateria": ["battery","charge","charging","batteries"],
  "pantalla": ["screen","display","resolution","touchscreen","lcd"],
  "precio": ["price","cost","expensive","cheap","overpriced"],
  "calidad": ["quality","build","durable","sturdy","flimsy"],
  "facilidad_uso": ["easy to use","intuitive","user-friendly","setup","install"],
  "envio_embalaje": ["shipping","packaging","delivery","packaged"],
  "atencion_cliente": ["customer service","warranty","refund","support"],
  "rendimiento": ["performance","fast","slow","lag","responsive"],
}
ASP_EN = {"bateria":"battery","pantalla":"screen","precio":"price","calidad":"build quality",
  "facilidad_uso":"ease of use","envio_embalaje":"shipping","atencion_cliente":"customer service","rendimiento":"performance"}
pares = []
for asp, kws in ASPECTOS.items():
    patron = r"(?:" + "|".join(re.escape(k) for k in kws) + r")"
    mask = low.str.contains(patron, regex=True)
    for i in df.index[mask]: pares.append((i, asp))
print(f"Pares resena x aspecto: {len(pares):,}")
absa = pipeline("text-classification", model="yangheng/deberta-v3-base-absa-v1.1",
    tokenizer="yangheng/deberta-v3-base-absa-v1.1", device=0, truncation=True, max_length=256)
inputs = [f"{df.at[i,'texto']} [SEP] {ASP_EN[a]}" for (i,a) in pares]
t0=time.time(); out = absa(inputs, batch_size=128); mins=(time.time()-t0)/60
absa_df = pd.DataFrame({"idx":[p[0] for p in pares],"aspecto":[p[1] for p in pares],
    "sentimiento":[o["label"] for o in out],"score":[o["score"] for o in out]})
absa_df.to_parquet(f"{OUT}/resultados/absa_resultados_200k.parquet")
dist = absa_df["sentimiento"].value_counts()
print(f"ABSA: {len(inputs):,} pares en {mins:.1f} min"); print(dist)
tab = (absa_df.groupby("aspecto")["sentimiento"].value_counts(normalize=True).mul(100).round(1).unstack(fill_value=0))
for c in ["Negative","Neutral","Positive"]:
    if c not in tab.columns: tab[c]=0
tab = tab[["Negative","Neutral","Positive"]]
tab["menciones"]=absa_df.groupby("aspecto").size(); tab["neto"]=(tab["Positive"]-tab["Negative"]).round(1)
tab=tab.sort_values("neto"); print(tab.to_string()); tab.to_csv(f"{OUT}/resultados/absa_por_aspecto_200k.csv")
orden=tab.index.tolist(); fig,ax=plt.subplots(figsize=(11,6))
colores={"Negative":"#e15759","Neutral":"#bfbfbf","Positive":"#59a14f"}; izq=np.zeros(len(orden))
for s in ["Negative","Neutral","Positive"]:
    vals=tab.loc[orden,s].values; ax.barh(orden,vals,left=izq,color=colores[s],label=s)
    for i,(v,l) in enumerate(zip(vals,izq)):
        if v>=6: ax.text(l+v/2,i,f"{v:.0f}%",va="center",ha="center",color="white",fontsize=9,fontweight="bold")
    izq+=vals
ax.set_xlim(0,100); ax.set_xlabel("% de menciones")
ax.set_title("Opyna - Sentimiento por aspecto (Amazon Electronics, 200.000 resenas)",fontsize=13,fontweight="bold")
ax.legend(ncol=3,loc="lower center",bbox_to_anchor=(0.5,-0.15),frameon=False)
for i,asp in enumerate(orden): ax.text(101,i,f"n={int(tab.loc[asp,'menciones'])}",va="center",fontsize=8,color="#555")
plt.tight_layout(); plt.savefig(f"{OUT}/figuras/absa_por_aspecto_200k.png",dpi=130,bbox_inches="tight"); plt.show()
neto_txt = " | ".join([f"{a} {tab.loc[a,'neto']:+.0f}" for a in orden])
mem("## 5. ABSA - Sentimiento por aspecto (DeBERTa deberta-v3-base-absa-v1.1)\n"
    f"- {len(inputs):,} pares resena x aspecto en {mins:.1f} min (A100).\n"
    "- Distribucion global: " + ", ".join([f"{k} {v:,}" for k,v in dist.items()]) + ".\n"
    f"- Sentimiento neto (Pos-Neg) por aspecto: {neto_txt}.\n"
    f"```\n{tab.to_string()}\n```\n"
    "- Guardado: `resultados/absa_resultados_200k.parquet`, `resultados/absa_por_aspecto_200k.csv`, `figuras/absa_por_aspecto_200k.png`\n")
print("PASO 3 OK -> Drive")

## 5. Agregacion ABSA por aspecto

In [ ]:
# PASO 4: RECAP - memoria completa guardada en Drive + arbol de archivos
import os
print(open(MEM, encoding="utf-8").read())
print("\n=== ARCHIVOS EN OUT (Opyna_TFM) ===")
for root,_,files in sorted(os.walk(OUT)):
    for f in sorted(files):
        p = os.path.join(root,f)
        try: sz = os.path.getsize(p)/1e6
        except: sz = 0
        print(f"  {p.replace(OUT,'Opyna_TFM')}  ({sz:.1f} MB)")

## 6. Gold set para validacion

In [ ]:
# PASO 5 (VM nueva): montar Drive + (a) metrica SENTIMIENTO + (b) GOLD SET ABSA para validacion humana
import pandas as pd, numpy as np
OUT = OUT
print("=== SENTIMIENTO 200k ===")
print(open(f"{OUT}/resultados/sentimiento_200k.txt").read())

# --- Gold set ABSA: muestra estratificada (20 por aspecto) para etiquetar a mano ---
absa = pd.read_parquet(f"{OUT}/resultados/absa_resultados_200k.parquet")
rev = pd.read_parquet(f"{OUT}/resultados/reviews_electronics_200k.parquet").reset_index(drop=True)
rev["texto"] = (rev["title"].fillna("")+" "+rev["text"].fillna("")).str.strip()
absa = absa.merge(rev[["texto"]], left_on="idx", right_index=True, how="left")
parts=[]
for asp, g in absa.groupby("aspecto"):
    parts.append(g.sample(min(20,len(g)), random_state=42))
gold = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
gold["texto_corto"] = gold["texto"].str.slice(0,300)
gold[["texto_corto","aspecto"]].assign(sentimiento_humano="").to_csv(f"{OUT}/resultados/absa_goldset_etiquetar_200k.csv", index=False)
gold[["texto_corto","aspecto","sentimiento","score"]].to_csv(f"{OUT}/resultados/absa_goldset_clave_200k.csv", index=False)
print(f"\nGOLD SET ABSA: {len(gold)} pares para etiquetar -> resultados/absa_goldset_etiquetar_200k.csv")
print("Clave del modelo (comparar tras etiquetar) -> resultados/absa_goldset_clave_200k.csv")
print(gold["aspecto"].value_counts().to_string())
MEM=f"{OUT}/memoria_resultados_200k.md"
open(MEM,"a").write("\n## 6. Validacion ABSA (gold set)\n"
    f"- Muestra estratificada de {len(gold)} pares (20/aspecto) exportada para etiquetado humano ciego.\n"
    "- Ficheros: `resultados/absa_goldset_etiquetar_200k.csv` (a rellenar) y `absa_goldset_clave_200k.csv` (prediccion del modelo).\n"
    "- Pendiente: etiquetar y calcular accuracy/F1 del ABSA frente al humano.\n")
print("PASO 5 OK -> Drive")

## 7. Validacion del ABSA

In [ ]:
# PASO 6: VALIDACION ABSA - DeBERTa vs anotador LLM (Claude), keyed-by-index (robusto)
import pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
OUT=OUT
clave = pd.read_csv(f"{OUT}/resultados/absa_goldset_clave_200k.csv")
claude_labels = {0:"Negative",1:"Negative",2:"Negative",3:"Neutral",4:"Negative",5:"Negative",6:"Negative",7:"Neutral",8:"Negative",9:"Positive",10:"Neutral",11:"Positive",12:"Negative",13:"Negative",14:"Positive",15:"Positive",16:"Negative",17:"Negative",18:"Negative",19:"Negative",20:"Positive",21:"Negative",22:"Negative",23:"Negative",24:"Positive",25:"Neutral",26:"Negative",27:"Negative",28:"Negative",29:"Positive",30:"Positive",31:"Neutral",32:"Neutral",33:"Negative",34:"Neutral",35:"Negative",36:"Negative",37:"Positive",38:"Positive",39:"Positive",40:"Negative",41:"Neutral",42:"Negative",43:"Negative",44:"Negative",45:"Negative",46:"Neutral",47:"Neutral",48:"Negative",49:"Negative",50:"Positive",51:"Positive",52:"Positive",53:"Positive",54:"Negative",55:"Negative",56:"Positive",57:"Positive",58:"Neutral",59:"Negative",60:"Positive",61:"Neutral",62:"Positive",63:"Negative",64:"Negative",65:"Positive",66:"Negative",67:"Negative",68:"Negative",69:"Negative",70:"Neutral",71:"Neutral",72:"Positive",73:"Positive",74:"Neutral",75:"Negative",76:"Positive",77:"Negative",78:"Positive",79:"Positive",80:"Negative",81:"Positive",82:"Neutral",83:"Positive",84:"Negative",85:"Positive",86:"Negative",87:"Positive",88:"Negative",89:"Positive",90:"Neutral",91:"Neutral",92:"Negative",93:"Negative",94:"Negative",95:"Negative",96:"Negative",97:"Positive",98:"Negative",99:"Positive",100:"Negative",101:"Negative",102:"Negative",103:"Negative",104:"Positive",105:"Neutral",106:"Negative",107:"Positive",108:"Negative",109:"Neutral",110:"Negative",111:"Positive",112:"Positive",113:"Neutral",114:"Positive",115:"Negative",116:"Negative",117:"Negative",118:"Positive",119:"Negative",120:"Positive",121:"Negative",122:"Negative",123:"Positive",124:"Negative",125:"Neutral",126:"Negative",127:"Positive",128:"Negative",129:"Positive",130:"Positive",131:"Neutral",132:"Positive",133:"Negative",134:"Positive",135:"Neutral",136:"Negative",137:"Positive",138:"Negative",139:"Negative",140:"Positive",141:"Negative",142:"Negative",143:"Positive",144:"Neutral",145:"Negative",146:"Negative",147:"Negative",148:"Positive",149:"Neutral",150:"Positive",151:"Neutral",152:"Positive",153:"Neutral",154:"Neutral",155:"Positive",156:"Positive",157:"Neutral",158:"Negative",159:"Negative"}
idxs = sorted(claude_labels.keys())
sub = clave.iloc[idxs].copy(); sub["claude"] = [claude_labels[i] for i in idxs]
print("SPOT-CHECK (150 y 155 deben salir Positive/Positive):")
for i in [9,20,29,99,150,155,159]:
    r=clave.iloc[i]; print(i,"|",r["aspecto"],"|",str(r["texto_corto"])[:48],"| claude=",claude_labels[i],"| deberta=",r["sentimiento"])
y_model=sub["sentimiento"].astype(str).values; y_claude=sub["claude"].values
acc=accuracy_score(y_claude,y_model); f1=f1_score(y_claude,y_model,average="macro")
labs=["Negative","Neutral","Positive"]
rep=classification_report(y_claude,y_model,labels=labs,digits=3,zero_division=0)
cm=confusion_matrix(y_claude,y_model,labels=labs)
print(f"\nn={len(idxs)} | Acuerdo(acc)={acc:.3f} | F1-macro={f1:.3f}")
print(rep); print("Matriz (filas=Claude, cols=DeBERTa; Neg/Neu/Pos):"); print(cm)
sub["ok"]=(y_model==y_claude); print("\nAcuerdo por aspecto:"); print(sub.groupby("aspecto")["ok"].mean().round(3).to_string())
open(f"{OUT}/resultados/absa_validacion_200k.txt","w").write(f"Validacion ABSA (DeBERTa vs Claude LLM-annotator, n={len(idxs)})\nAccuracy={acc:.4f} F1m={f1:.4f}\n\n{rep}\n\nMatriz (Claude filas/DeBERTa cols; Neg/Neu/Pos):\n{cm}\n")
open(f"{OUT}/memoria_resultados_200k.md","a").write("\n## 7. Validacion ABSA vs anotador LLM (Claude)\n"
    f"- {len(idxs)} pares etiquetados por Claude como anotador independiente (LLM-as-annotator, NO ground-truth humano).\n"
    f"- **Acuerdo con DeBERTa: accuracy {acc:.3f} / F1-macro {f1:.3f}**.\n"
    "- Validacion orientativa por LLM; para rigor pleno se recomienda ademas etiquetado humano.\n"
    f"```\n{rep}\n```\n")
print("VALIDACION OK -> Drive")

## 8. Modelo predictivo de utilidad (helpful)

In [ ]:
# PASO 7b - helpful balanceado + calibrado + top-tokens -> Drive + memoria
import os, glob, json, joblib, numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score,average_precision_score,brier_score_loss,confusion_matrix,classification_report
BASE=OUT; SEED=42
for s in ['modelos','resultados','figuras']: os.makedirs(BASE+'/'+s, exist_ok=True)
cand=[c for c in glob.glob(BASE+'/**/*.parquet', recursive=True) if 'reviews' in c.lower()]
df=pd.read_parquet(cand[0])
TXT='text' if 'text' in df.columns else 'texto'
df['t']=(df['title'].fillna('')+' '+df[TXT].fillna('')).str.strip()
y=(df['helpful_vote'].fillna(0).astype(int)>=1).astype(int)
print('base_rate y=1:', round(float(y.mean()),3))
NUM=pd.DataFrame({'lw':df['t'].str.split().str.len().clip(0,400),'lc':df['t'].str.len().clip(0,4000),'rating':df['rating'].astype(float),'verif':df['verified_purchase'].astype(float)})
NUM=(NUM-NUM.mean())/(NUM.std()+1e-9)
Xt,Xe,Nt,Ne,yt,ye=train_test_split(df['t'],NUM.values,y,test_size=0.2,random_state=SEED,stratify=y)
vec=TfidfVectorizer(min_df=5,ngram_range=(1,2),max_features=50000,sublinear_tf=True)
Xtr=hstack([vec.fit_transform(Xt),csr_matrix(Nt)]).tocsr()
Xte=hstack([vec.transform(Xe),csr_matrix(Ne)]).tocsr()
base=LogisticRegression(max_iter=1000,class_weight='balanced',n_jobs=-1)
clf=CalibratedClassifierCV(base,method='isotonic',cv=3).fit(Xtr,yt)
pr=clf.predict_proba(Xte)[:,1]; yp=(pr>=0.5).astype(int)
acc=accuracy_score(ye,yp); f1=f1_score(ye,yp,average='macro'); auc=roc_auc_score(ye,pr); ap=average_precision_score(ye,pr); br=brier_score_loss(ye,pr)
print('BAL+CALIB acc=%.3f f1m=%.3f roc_auc=%.3f pr_auc=%.3f brier=%.3f'%(acc,f1,auc,ap,br))
print(confusion_matrix(ye,yp)); print(classification_report(ye,yp,digits=3))
it=LogisticRegression(max_iter=1000,class_weight='balanced',n_jobs=-1).fit(Xtr,yt)
fn=np.array(vec.get_feature_names_out().tolist()+list(NUM.columns)); co=it.coef_[0]; o=np.argsort(co)
print('TOP util:', [fn[i] for i in o[-12:][::-1]])
print('TOP no-util:', [fn[i] for i in o[:12]])
joblib.dump({'vectorizer':vec,'model':clf}, BASE+'/modelos/helpful_200k.joblib')
json.dump({'acc':acc,'f1m':f1,'auc':auc,'ap':ap,'brier':br,'base_rate':float(y.mean())}, open(BASE+'/resultados/helpful_200k.txt','w'), indent=2)
L=['','## 8. Modelo predictivo utilidad (helpful, balanceado + calibrado)','- Target helpful_vote>=1 (base %.1f%%). TF-IDF+meta+LogReg balanceada, calibrada isotonica.'%(100*y.mean()),'- Test: acc %.3f | F1m %.3f | ROC-AUC %.3f | PR-AUC %.3f | Brier %.3f.'%(acc,f1,auc,ap,br),'- Artefactos: modelos/helpful_200k.joblib, resultados/helpful_200k.txt.']
open(BASE+'/memoria_resultados_200k.md','a').write(chr(10).join(L)+chr(10))
print('OK Paso7 balanceado -> Drive')

## 9. Temas y problemas emergentes (BERTopic)

In [ ]:
# PASO 8 - Temas/problemas emergentes (BERTopic lean) sobre resenas negativas -> Drive + memoria
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','bertopic','sentence-transformers'], check=True)
import os, glob, pandas as pd, numpy as np, matplotlib.pyplot as plt
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
BASE=OUT; SEED=42
for s in ['modelos','resultados','figuras']: os.makedirs(BASE+'/'+s, exist_ok=True)
cand=[c for c in glob.glob(BASE+'/**/*.parquet', recursive=True) if 'reviews' in c.lower()]
df=pd.read_parquet(cand[0]); TXT='text' if 'text' in df.columns else 'texto'
df['t']=(df['title'].fillna('')+' '+df[TXT].fillna('')).str.strip()
neg=df[df['rating']<=2]['t']; neg=neg[neg.str.split().str.len()>=5]
N=min(15000,len(neg)); docs=neg.sample(N, random_state=SEED).tolist()
print('resenas negativas para BERTopic:', len(docs))
emb=SentenceTransformer('all-MiniLM-L6-v2')
E=emb.encode(docs, batch_size=256, show_progress_bar=True)
vzr=CountVectorizer(stop_words='english', ngram_range=(1,2), min_df=2)
tm=BERTopic(embedding_model=emb, vectorizer_model=vzr, min_topic_size=80, nr_topics='auto', calculate_probabilities=False, language='english', verbose=True)
topics,_=tm.fit_transform(docs, E)
info=tm.get_topic_info(); info=info[info.Topic!=-1]
print('topicos (sin outliers):', len(info))
def terms(t,n=5): return ', '.join(w for w,_ in tm.get_topic(t)[:n])
top=info.sort_values('Count', ascending=False).head(10).copy()
top['Terminos']=top['Topic'].apply(terms)
print('TOP TEMAS EN RESENAS NEGATIVAS:')
for _,r in top.iterrows(): print(' [%d] %s'%(int(r.Count), r.Terminos))
asp={'bateria':['battery','charge','charging'],'pantalla':['screen','display','touch'],'precio':['price','money','expensive','cheap'],'calidad':['quality','flimsy','broke','broken'],'envio':['shipping','delivery','arrived','package'],'rendimiento':['slow','lag','performance','crash'],'facilidad_uso':['easy','setup','install','instructions'],'atencion_cliente':['support','service','refund','return','warranty','seller']}
def amap(tt):
  tl=tt.lower(); h=[a for a,k in asp.items() if any(x in tl for x in k)]; return ', '.join(h) if h else '-'
top['Aspecto']=top['Terminos'].apply(amap)
fig,ax=plt.subplots(figsize=(9,5)); ax.barh(list(top['Terminos'])[::-1], list(top['Count'])[::-1])
ax.set_title('Temas en resenas negativas (BERTopic)'); ax.set_xlabel('n resenas'); plt.tight_layout()
fig.savefig(BASE+'/figuras/temas_emergentes_200k.png', dpi=130)
top[['Topic','Count','Terminos','Aspecto']].to_csv(BASE+'/resultados/temas_emergentes_200k.csv', index=False)
tm.save(BASE+'/modelos/bertopic_negativas_200k', serialization='safetensors', save_ctfidf=True, save_embedding_model='all-MiniLM-L6-v2')
L=['','## 9. Temas/problemas emergentes (BERTopic) - cumple 5.3 de la propuesta','- Fase de inteligencia (no supervisada): agrupamiento de resenas negativas (rating<=2, n=%d).'%len(docs)]
for _,r in top.iterrows(): L.append('- %s (%d resenas) -> aspecto: %s'%(r.Terminos, int(r.Count), r.Aspecto))
L.append('- Caveat: muestra balanceada por estrellas, no es corte temporal; seguimiento temporal como linea futura.')
L.append('- Artefactos: modelos/bertopic_negativas_200k, figuras/temas_emergentes_200k.png, resultados/temas_emergentes_200k.csv.')
open(BASE+'/memoria_resultados_200k.md','a').write(chr(10).join(L)+chr(10))
print('OK Paso8 BERTopic -> Drive')

## 10. Interpretabilidad (SHAP + atencion)

In [ ]:
# PASO 10 - Interpretabilidad (6): SHAP (modelo lineal) + atencion DistilBERT -> Drive + memoria
import os, glob, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
try:
    import shap
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','shap'],check=True); import shap
BASE=OUT; SEED=42
cand=[c for c in glob.glob(BASE+'/**/*.parquet', recursive=True) if 'reviews' in c.lower()]
df=pd.read_parquet(cand[0]); TXT='text' if 'text' in df.columns else 'texto'
df['texto']=(df['title'].fillna('')+' '+df[TXT].fillna('')).str.strip()
df['y']=df['rating'].apply(lambda r: 0 if r<=2 else (1 if r==3 else 2))
Xtr,Xte,ytr,yte=train_test_split(df['texto'], df['y'], test_size=0.2, random_state=SEED, stratify=df['y'])
vec=TfidfVectorizer(ngram_range=(1,2), min_df=5, max_features=20000, sublinear_tf=True)
Xtr_v=vec.fit_transform(Xtr); Xte_v=vec.transform(Xte)
model=LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1).fit(Xtr_v, ytr)
feat=np.array(vec.get_feature_names_out())
rs=np.random.RandomState(SEED)
bg=Xtr_v[rs.choice(Xtr_v.shape[0],300,replace=False)].toarray()
samp=Xte_v[rs.choice(Xte_v.shape[0],800,replace=False)].toarray()
expl=shap.LinearExplainer(model, bg)
sv=expl.shap_values(samp)
clases=['negativo','neutral','positivo']
def topk(k,n=12):
    vals = sv[k] if isinstance(sv,list) else (sv[:,:,k] if getattr(sv,'ndim',2)==3 else sv)
    imp=np.abs(vals).mean(axis=0); idx=np.argsort(imp)[::-1][:n]; return [feat[i] for i in idx]
print('== SHAP top tokens por clase ==')
shap_lines=[]
for k,c in enumerate(clases):
    tk=topk(k); print(c,'->',tk); shap_lines.append(c+': '+', '.join(tk))
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tkz=AutoTokenizer.from_pretrained(BASE+'/modelos/sentimiento_200k')
mdl=AutoModelForSequenceClassification.from_pretrained(BASE+'/modelos/sentimiento_200k', output_attentions=True); mdl.eval()
ejemplos=["The battery drains way too fast and the screen is dim, very disappointed.","Excellent sound quality and easy setup, works perfectly."]
att_lines=[]
for ex in ejemplos:
    enc=tkz(ex, return_tensors='pt', truncation=True, max_length=64)
    with torch.no_grad(): out=mdl(**enc)
    att=out.attentions[-1][0].mean(0); cls_att=att[0]
    toks=tkz.convert_ids_to_tokens(enc['input_ids'][0])
    order=cls_att.argsort(descending=True).tolist()
    top=[toks[i] for i in order if toks[i] not in ['[CLS]','[SEP]']][:6]
    pred=clases[int(out.logits.argmax())]
    print('EJ:', ex[:50], '| pred=', pred, '| atiende:', top)
    att_lines.append(ex[:60]+' -> pred '+pred+' | atiende: '+', '.join(top))
L=['','## 11. Interpretabilidad (6): SHAP (lineal) + atencion (DistilBERT)','- SHAP (LinearExplainer) sobre TF-IDF(20k)+LogReg; tokens mas influyentes por clase:']
for s in shap_lines: L.append('  - '+s)
L.append('- Atencion del transformer (ultima capa, CLS->tokens) en ejemplos:')
for a in att_lines: L.append('  - '+a)
open(BASE+'/memoria_resultados_200k.md','a').write(chr(10).join(L)+chr(10))
print('OK Paso10 interpretabilidad -> Drive')

## 11. Capa generativa de recomendaciones

In [ ]:
# PASO 11b - Generar recomendaciones.json (import correcto) + memoria
import importlib.util, sys
BASE=OUT
spec=importlib.util.spec_from_file_location('opyna_reco', BASE+'/opyna_recomendaciones.py')
mod=importlib.util.module_from_spec(spec); sys.modules['opyna_reco']=mod; spec.loader.exec_module(mod)
recs=mod.priorizar(); mod.exportar_json(recs, BASE+'/resultados/recomendaciones.json')
print('Priorizacion determinista:')
for r in recs: print('  prio', r.prioridad, '|', r.aspecto, '|', r.tipo, '| neto', r.neto)
import os
mem=BASE+'/memoria_resultados_200k.md'
ya = os.path.exists(mem) and '## 12. Capa generativa' in open(mem,encoding='utf-8').read()
if not ya:
    L=['','## 12. Capa generativa de recomendaciones (2a)','- Priorizacion determinista (menciones x |neto negativo|) + capa LLM opcional (Gemini).','- Entregables: resultados/recomendaciones.md, resultados/recomendaciones.json, opyna_recomendaciones.py.']
    open(mem,'a').write(chr(10).join(L)+chr(10))
    print('memoria: seccion 12 anadida')
else:
    print('memoria: seccion 12 ya existia')
print('OK Paso11 recomendaciones -> Drive')

## 12. ABSA + recomendaciones de negocio

In [ ]:
# PASO 13 - Dashboard Gradio (demo A): modelos de Drive + enlace publico
import subprocess, sys, importlib.util, torch
subprocess.run([sys.executable,'-m','pip','install','-q','gradio','sentencepiece'], check=True)
BASE=OUT
import gradio as gr
from transformers import pipeline
DEV=0 if torch.cuda.is_available() else -1
sent_pipe=pipeline('text-classification', model=BASE+'/modelos/sentimiento_200k', device=DEV, truncation=True, max_length=256)
rate_pipe=pipeline('text-classification', model=BASE+'/modelos/rating_200k', device=DEV, truncation=True, max_length=256)
absa_pipe=pipeline('text-classification', model='yangheng/deberta-v3-base-absa-v1.1', device=DEV, truncation=True, max_length=256)
spec=importlib.util.spec_from_file_location('opyna_reco', BASE+'/opyna_recomendaciones.py')
reco=importlib.util.module_from_spec(spec); sys.modules['opyna_reco']=reco; spec.loader.exec_module(reco)
ASPECTOS={'bateria':['battery','charge','charging','batteries'],'pantalla':['screen','display','resolution','touchscreen','lcd'],'precio':['price','cost','expensive','cheap','overpriced'],'calidad':['quality','build','durable','sturdy','flimsy'],'facilidad_uso':['easy to use','intuitive','user-friendly','setup','install'],'envio_embalaje':['shipping','packaging','delivery','packaged'],'atencion_cliente':['customer service','warranty','refund','support'],'rendimiento':['performance','fast','slow','lag','responsive']}
ASP_EN={'bateria':'battery','pantalla':'screen','precio':'price','calidad':'build quality','facilidad_uso':'ease of use','envio_embalaje':'shipping','atencion_cliente':'customer service','rendimiento':'performance'}
SENT={'LABEL_0':'Negativo','LABEL_1':'Neutral','LABEL_2':'Positivo'}
def analizar(texto):
    if not texto or len(texto.strip())<3: return 'Escribe una resena (en ingles).'
    t=texto.strip()
    s=sent_pipe(t)[0]; senti=SENT.get(s['label'],s['label'])
    r=rate_pipe(t)[0]; estrellas=int(r['label'].split('_')[-1])+1
    low=t.lower(); L=['## Resultado']
    L.append('- Sentimiento global: %s (%.0f%%)'%(senti,100*s['score']))
    L.append('- Rating estimado: %d de 5 estrellas'%estrellas)
    filas=[]; negativos=[]
    for asp,kws in ASPECTOS.items():
        if any(k in low for k in kws):
            out=absa_pipe(t+' [SEP] '+ASP_EN[asp])[0]
            filas.append((asp,out['label']))
            if out['label']=='Negative': negativos.append(asp)
    if filas:
        L.append(''); L.append('### Aspectos detectados')
        for a,l in filas: L.append('- %s -> %s'%(a,l))
    else:
        L.append(''); L.append('_No se detectaron aspectos concretos._')
    if negativos:
        L.append(''); L.append('### Recomendaciones')
        for a in negativos: L.append('- %s: %s'%(a, reco.ACCIONES.get(a,'')))
    return chr(10).join(L)
demo=gr.Interface(fn=analizar, inputs=gr.Textbox(lines=4, label='Resena (en ingles)'), outputs=gr.Markdown(), title='Opyna - Inteligencia de Voz del Cliente', description='Pega una resena de producto y obten sentimiento, rating, aspectos y recomendaciones.', examples=[['The battery drains way too fast and customer service was useless.'],['Great screen and very easy to set up, works perfectly.']])
demo.launch(share=True)
print('OK Paso13 demo Gradio lanzada')

## 13. Caso de negocio y ROI

In [ ]:
# PASO 12 - Exportar caso de negocio / ROI (seccion 8) -> Drive + memoria
import base64, os, importlib.util, sys, json
BASE=OUT
os.makedirs(BASE+'/resultados', exist_ok=True)
B64_MD='IyBPcHluYSDigJQgQ2FzbyBkZSBuZWdvY2lvIHkgUk9JCgoqTW9kZWxvIGlsdXN0cmF0aXZvIGNvbiBzdXB1ZXN0b3MgY29uc2VydmFkb3JlcyB5IGRlY2xhcmFkb3MuIEVsIG9iamV0aXZvIG5vIGVzIHVuIG7Dum1lcm8gZXhhY3RvLCBzaW5vIG1vc3RyYXIgZWwgb3JkZW4gZGUgbWFnbml0dWQgZGVsIHJldG9ybm8geSBxdWUgbGEgbMOzZ2ljYSBlcyBkZWZlbmRpYmxlIHkgYWp1c3RhYmxlICh2ZXIgYG9weW5hX3JvaS5weWApLioKCiMjIEVsIHByb2JsZW1hLCBlbiBuw7ptZXJvcwoKVW4gdmVuZGVkb3IgZGUgZWxlY3Ryw7NuaWNhIGRlIHRhbWHDsW8gbWVkaW8gcmVjaWJlIGRlY2VuYXMgZGUgbWlsZXMgZGUgcmVzZcOxYXMgYWwgYcOxbyBlbiB0ZXh0byBsaWJyZS4gTGVlcmxhcyB5IGNsYXNpZmljYXJsYXMgYSBtYW5vIHBvciBhc3BlY3RvIChwcmVjaW8sIGJhdGVyw61hLCBwYW50YWxsYSwgc29wb3J0ZeKApikgbm8gZXNjYWxhOiBhIHVuIHJpdG1vIHJlYWxpc3RhIGRlIH4zMCByZXNlw7Fhcy9ob3JhIGNvbiBldGlxdWV0YWRvIGN1aWRhZG9zbywgMTIwLjAwMCByZXNlw7Fhcy9hw7FvIHNvbiB+NC4wMDAgaG9yYXMgZGUgYW5hbGlzdGEuIEVuIGxhIHByw6FjdGljYSwgbGFzIGVtcHJlc2FzICoqbm8gbGFzIGxlZW4gdG9kYXMqKiDigJRtdWVzdHJlYW4gbyBsYXMgaWdub3JhbuKAlCB5IHBpZXJkZW4gbGEgc2XDsWFsIGp1c3RvIGRvbmRlIGVzdMOhIGVsIHByb2JsZW1hLiBMYXMgaGVycmFtaWVudGFzIGNvbWVyY2lhbGVzIGRlIFZvaWNlIG9mIEN1c3RvbWVyIGN1YnJlbiBlc2UgaHVlY28sIHBlcm8gYSB1biBjb3N0ZSBkZSBkZWNlbmFzIGRlIG1pbGVzIGRlIGV1cm9zIGFsIGHDsW8uCgojIyBEb3MgcGFsYW5jYXMgZGUgdmFsb3IKCioqMS4gQWhvcnJvIGVuIGVsIGFuw6FsaXNpcy4qKiBPcHluYSBzdXN0aXR1eWUgZWwgYW7DoWxpc2lzIG1hbnVhbC9TYWFTIHBvciB1biBwaXBlbGluZSBhdXRvbcOhdGljbyBjdXlvIGNvc3RlIG1hcmdpbmFsIGVzIG3DrW5pbW8gKGPDs21wdXRvICsgaG9zdGluZyArIHVuYSBBUEkgZGUgTExNIG9wY2lvbmFsKS4gRXMgZWwgYWhvcnJvIG3DoXMgKipyb2J1c3RvIHkgZGlyZWN0byoqLgoKKioyLiBSZWR1Y2Npw7NuIGRlIGRldm9sdWNpb25lcyB5IGNodXJuLioqIEVzIGxhIHBhbGFuY2EgZ3JhbmRlLCBwZXJvIGRlcGVuZGUgZGUgcXVlIGxhIG1hcmNhICoqYWN0w7plKiogc29icmUgbG8gcXVlIE9weW5hIHByaW9yaXphLiBFbiBlc3RlIGRhdGFzZXQsIGVsIHByb2JsZW1hIG7CujEgZXMgKiphdGVuY2nDs24gYWwgY2xpZW50ZSoqIChuZXRvIOKIkjQ4KSwgYWxpbWVudGFkbyBwb3IgZWwgdGVtYSAqImRlasOzIGRlIGZ1bmNpb25hciBhIGxvcyBwb2NvcyBtZXNlcyIqIChkdXJhYmlsaWRhZCDihpIgZGV2b2x1Y2lvbmVzIHkgZ2FyYW50w61hcykuIEFjdHVhciBhaMOtIHJlZHVjZSBkZXZvbHVjaW9uZXMgeSBwcm90ZWdlIGxhIHJldGVuY2nDs24uCgojIyBTdXB1ZXN0b3MgZGVsIGVzY2VuYXJpbyBiYXNlIChkZWNsYXJhZG9zKQoKfCBQYXLDoW1ldHJvIHwgVmFsb3IgfCBDb21lbnRhcmlvIHwKfC0tLXwtLS18LS0tfAp8IEluZ3Jlc29zIGFudWFsZXMgfCAxMC4wMDAuMDAwIOKCrCB8IFZlbmRlZG9yIGRlIHRhbWHDsW8gbWVkaW8gfAp8IFBlZGlkbyBtZWRpbyB8IDUwIOKCrCB8IOKGkiAyMDAuMDAwIHBlZGlkb3MvYcOxbyB8CnwgUmVzZcOxYXMvYcOxbyB8IDEyMC4wMDAgfCB+MTAuMDAwL21lcyB8CnwgVGFzYSBkZSBkZXZvbHVjacOzbiB8IDEwICUgfCDihpIgMjAuMDAwIGRldm9sdWNpb25lcy9hw7FvIHwKfCBDb3N0ZSBwb3IgZGV2b2x1Y2nDs24gfCA0MCDigqwgfCBMb2fDrXN0aWNhICsgaW5zcGVjY2nDs24gKyBww6lyZGlkYSBkZSBtYXJnZW4gfAp8IENvc3RlIGRldm9sdWNpb25lcy9hw7FvIHwgODAwLjAwMCDigqwgfCBCYXNlIHNvYnJlIGxhIHF1ZSBzZSBhY3TDumEgfAp8IFJlZHVjY2nDs24gZGUgZGV2b2x1Y2lvbmVzIHbDrWEgT3B5bmEgfCAxMCAlIHJlbGF0aXZvIHwgQWwgcHJpb3JpemFyIHkgY29ycmVnaXIgbG9zIHByb2JsZW1hcyB0b3AgfAp8IENvc3RlIFZvQyBldml0YWRvIChTYWFTL21hbnVhbCkgfCAyNS4wMDAg4oKsL2HDsW8gfCBBbHRlcm5hdGl2YSBxdWUgT3B5bmEgc3VzdGl0dXllIHwKfCBDb3N0ZSBkZSBvcGVyYXIgT3B5bmEgfCB+MS4wMDAg4oKsL2HDsW8gfCBDw7NtcHV0byArIGhvc3RpbmcgKyBMTE0gKGVsIGRlc2Fycm9sbG8gZnVlIGVsIFRGTSwgPDEwMCDigqwpIHwKCiMjIFJlc3VsdGFkbyDigJQgZXNjZW5hcmlvIGJhc2UKCi0gKipBaG9ycm8gcG9yIG1lbm9zIGRldm9sdWNpb25lczoqKiAyMC4wMDAgw5cgMTAgJSDDlyA0MCDigqwgPSAqKjgwLjAwMCDigqwvYcOxbyoqCi0gKipBaG9ycm8gZW4gYW7DoWxpc2lzIChWb0MgZXZpdGFkbyk6KiogMjUuMDAwIOKIkiAxLjAwMCA9ICoqMjQuMDAwIOKCrC9hw7FvKioKLSAqKlZhbG9yIHRvdGFsOioqIOKJiCAqKjEwNC4wMDAg4oKsL2HDsW8qKiDCtyAqKkNvc3RlOioqIOKJiCAxLjAwMCDigqwvYcOxbwotICoqUk9JIOKJiCAxMDDDlyAocGF5YmFjayBlbiBkw61hcykuKiogQXVuIHNpZW5kbyAxMCB2ZWNlcyBkZW1hc2lhZG8gb3B0aW1pc3RhcywgZWwgcmV0b3JubyBzaWd1ZSBzaWVuZG8gY2xhcmFtZW50ZSBwb3NpdGl2by4KCiMjIFNlbnNpYmlsaWRhZAoKfCBFc2NlbmFyaW8gfCBSZWR1Y2Npw7NuIGRldm9sdWNpb25lcyB8IEFob3JybyBhbsOhbGlzaXMgfCBSZXRlbmNpw7NuIHwgVmFsb3IgYW51YWwgfAp8LS0tfC0tLXwtLS18LS0tfC0tLXwKfCBDb25zZXJ2YWRvciB8IDUgJSAoNDAuMDAwIOKCrCkgfCAwIOKCrCAobm8gaGFiw61hIFZvQykgfCDigJQgfCAqKn40MC4wMDAg4oKsKiogfAp8IEJhc2UgfCAxMCAlICg4MC4wMDAg4oKsKSB8IDI0LjAwMCDigqwgfCDigJQgfCAqKn4xMDQuMDAwIOKCrCoqIHwKfCBPcHRpbWlzdGEgfCAxNSAlICgxMjAuMDAwIOKCrCkgfCAyNC4wMDAg4oKsIHwgKzAsNSAlIGNvbnZlcnNpw7NuICh+NTAuMDAwIOKCrCkgfCAqKn4xOTQuMDAwIOKCrCoqIHwKCkluY2x1c28gZW4gZWwgZXNjZW5hcmlvIGNvbnNlcnZhZG9yLCBlbCB2YWxvciAofjQwLjAwMCDigqwpIGZyZW50ZSBhIHVuIGNvc3RlIGRlIH4xLjAwMCDigqwgZGVqYSB1biByZXRvcm5vIGRlIGRvcyDDs3JkZW5lcyBkZSBtYWduaXR1ZC4KCiMjIFF1w6kgYXBvcnRhIE9weW5hICh5IHF1w6kgbm8pCgpPcHluYSAqKmlkZW50aWZpY2EgeSBwcmlvcml6YSBkw7NuZGUgYWN0dWFyKiog4oCUcXXDqSBhc3BlY3RvLCByZXNwYWxkYWRvIHBvciB2b2x1bWVuIHkgc2VudGltaWVudG8sIHkgY29uIGVsIHByb2JsZW1hIGNvbmNyZXRvIGRldHLDoXMgdsOtYSBCRVJUb3BpY+KAlCB5IGF1dG9tYXRpemEgYSBjb3N0ZSBjYXNpIG51bG8gdW4gYW7DoWxpc2lzIHF1ZSBhIG1hbm8gbm8gZXNjYWxhLiAqKk5vIGFycmVnbGEgZWwgcHJvZHVjdG8gcG9yIHPDrSBtaXNtbyoqOiBsYSBwYWxhbmNhIGRlIGRldm9sdWNpb25lcyBzZSBtYXRlcmlhbGl6YSBzb2xvIHNpIGxhIG1hcmNhIGVqZWN1dGEgbGFzIG1lam9yYXMuIFBvciBlc28gZWwgY2FzbyBkZSBuZWdvY2lvIHNlcGFyYSBsYSBwYWxhbmNhIHJvYnVzdGEgKGFob3JybyBkZSBhbsOhbGlzaXMpIGRlIGxhIHBhbGFuY2EgZGUgYWx0byBpbXBhY3RvIHBlcm8gZGVwZW5kaWVudGUgZGUgZWplY3VjacOzbiAoZGV2b2x1Y2lvbmVzL3JldGVuY2nDs24pLgoKIyMgUmVwcm9kdWNpYmlsaWRhZAoKTG9zIG7Dum1lcm9zIHNhbGVuIGRlIHVuIG1vZGVsbyBwYXJhbWV0cml6YWJsZSAoYG9weW5hX3JvaS5weWApOiBiYXN0YSBjYW1iaWFyIGxvcyBzdXB1ZXN0b3MgKGluZ3Jlc29zLCB0YXNhIGRlIGRldm9sdWNpw7NuLCByZWR1Y2Npw7NuIGVzcGVyYWRh4oCmKSBwYXJhIHJlY2FsY3VsYXIgZWwgUk9JIHkgbGEgc2Vuc2liaWxpZGFkLiBFcyB1biBtb2RlbG8gdHJhbnNwYXJlbnRlLCBubyB1bmEgY2lmcmEgY2VycmFkYS4K'
B64_PY='IiIiT3B5bmEgLSBDYWxjdWxhZG9yYSBkZSBST0kgKGNhc28gZGUgbmVnb2Npbywgc2VjY2lvbiA4KS4KCk1vZGVsbyB0cmFuc3BhcmVudGUgeSBwYXJhbWV0cml6YWJsZTogYSBwYXJ0aXIgZGUgc3VwdWVzdG9zIGRlY2xhcmFkb3Mgc29icmUgZWwKdmVuZGVkb3IgKGluZ3Jlc29zLCBkZXZvbHVjaW9uZXMsIGNvc3RlIGRlIFZvQyBldml0YWRvIHkgY29zdGUgZGUgb3BlcmFyIE9weW5hKQpjYWxjdWxhIGVsIHZhbG9yIGFudWFsLCBlbCBST0kgeSBlbCBwYXliYWNrLCBjb24gdHJlcyBlc2NlbmFyaW9zIGRlIHNlbnNpYmlsaWRhZC4KCk5vIHByZXRlbmRlIGRhciB1bmEgY2lmcmEgY2VycmFkYSwgc2lubyB1biBvcmRlbiBkZSBtYWduaXR1ZCBkZWZlbmRpYmxlOiBiYXN0YQpjYW1iaWFyIGxvcyBzdXB1ZXN0b3MgcGFyYSByZWNhbGN1bGFyLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBhc2RpY3QKCgpAZGF0YWNsYXNzCmNsYXNzIFN1cHVlc3RvczoKICAgIGluZ3Jlc29zX2FudWFsZXM6IGZsb2F0ID0gMTBfMDAwXzAwMC4wCiAgICBwZWRpZG9fbWVkaW86IGZsb2F0ID0gNTAuMAogICAgcmVzZW5hc19hbnVhbGVzOiBpbnQgPSAxMjBfMDAwCiAgICB0YXNhX2Rldm9sdWNpb246IGZsb2F0ID0gMC4xMCAgICAgICAgICAjIDEwJSBkZSBsb3MgcGVkaWRvcwogICAgY29zdGVfcG9yX2Rldm9sdWNpb246IGZsb2F0ID0gNDAuMCAgICAgIyBsb2dpc3RpY2EgKyBpbnNwZWNjaW9uICsgbWFyZ2VuIHBlcmRpZG8KICAgIHJlZHVjY2lvbl9kZXZvbHVjaW9uZXM6IGZsb2F0ID0gMC4xMCAgICMgcmVsYXRpdmEsIGFsIGFjdHVhciBzb2JyZSBsbyBwcmlvcml6YWRvCiAgICBjb3N0ZV92b2NfZXZpdGFkbzogZmxvYXQgPSAyNV8wMDAuMCAgICAjIFNhYVMvbWFudWFsIHF1ZSBPcHluYSBzdXN0aXR1eWUKICAgIGNvc3RlX29weW5hX2FudWFsOiBmbG9hdCA9IDFfMDAwLjAgICAgICMgY29tcHV0byArIGhvc3RpbmcgKyBMTE0KICAgIHVwbGlmdF9jb252ZXJzaW9uOiBmbG9hdCA9IDAuMCAgICAgICAgICMgcC5lai4gMC4wMDUgPSArMCw1JSBkZSBpbmdyZXNvcwoKCmRlZiBjYWxjdWxhcihzOiBTdXB1ZXN0b3MpIC0+IGRpY3Q6CiAgICBwZWRpZG9zID0gcy5pbmdyZXNvc19hbnVhbGVzIC8gcy5wZWRpZG9fbWVkaW8KICAgIGRldm9sdWNpb25lcyA9IHBlZGlkb3MgKiBzLnRhc2FfZGV2b2x1Y2lvbgogICAgY29zdGVfZGV2b2x1Y2lvbmVzID0gZGV2b2x1Y2lvbmVzICogcy5jb3N0ZV9wb3JfZGV2b2x1Y2lvbgogICAgYWhvcnJvX2Rldm9sdWNpb25lcyA9IGNvc3RlX2Rldm9sdWNpb25lcyAqIHMucmVkdWNjaW9uX2Rldm9sdWNpb25lcwogICAgYWhvcnJvX2FuYWxpc2lzID0gbWF4KHMuY29zdGVfdm9jX2V2aXRhZG8gLSBzLmNvc3RlX29weW5hX2FudWFsLCAwLjApCiAgICB2YWxvcl9yZXRlbmNpb24gPSBzLmluZ3Jlc29zX2FudWFsZXMgKiBzLnVwbGlmdF9jb252ZXJzaW9uCiAgICB2YWxvcl90b3RhbCA9IGFob3Jyb19kZXZvbHVjaW9uZXMgKyBhaG9ycm9fYW5hbGlzaXMgKyB2YWxvcl9yZXRlbmNpb24KICAgIGJlbmVmaWNpb19uZXRvID0gdmFsb3JfdG90YWwgLSBzLmNvc3RlX29weW5hX2FudWFsCiAgICByb2kgPSBiZW5lZmljaW9fbmV0byAvIHMuY29zdGVfb3B5bmFfYW51YWwgaWYgcy5jb3N0ZV9vcHluYV9hbnVhbCBlbHNlIGZsb2F0KCJpbmYiKQogICAgcGF5YmFja19kaWFzID0gMzY1ICogcy5jb3N0ZV9vcHluYV9hbnVhbCAvIHZhbG9yX3RvdGFsIGlmIHZhbG9yX3RvdGFsIGVsc2UgZmxvYXQoImluZiIpCiAgICByZXR1cm4gewogICAgICAgICJwZWRpZG9zIjogcm91bmQocGVkaWRvcyksCiAgICAgICAgImRldm9sdWNpb25lcyI6IHJvdW5kKGRldm9sdWNpb25lcyksCiAgICAgICAgImNvc3RlX2Rldm9sdWNpb25lcyI6IHJvdW5kKGNvc3RlX2Rldm9sdWNpb25lcyksCiAgICAgICAgImFob3Jyb19kZXZvbHVjaW9uZXMiOiByb3VuZChhaG9ycm9fZGV2b2x1Y2lvbmVzKSwKICAgICAgICAiYWhvcnJvX2FuYWxpc2lzIjogcm91bmQoYWhvcnJvX2FuYWxpc2lzKSwKICAgICAgICAidmFsb3JfcmV0ZW5jaW9uIjogcm91bmQodmFsb3JfcmV0ZW5jaW9uKSwKICAgICAgICAidmFsb3JfdG90YWwiOiByb3VuZCh2YWxvcl90b3RhbCksCiAgICAgICAgImJlbmVmaWNpb19uZXRvIjogcm91bmQoYmVuZWZpY2lvX25ldG8pLAogICAgICAgICJyb2lfeCI6IHJvdW5kKHJvaSwgMSksCiAgICAgICAgInBheWJhY2tfZGlhcyI6IHJvdW5kKHBheWJhY2tfZGlhcywgMSksCiAgICB9CgoKRVNDRU5BUklPUyA9IHsKICAgICJjb25zZXJ2YWRvciI6IFN1cHVlc3RvcyhyZWR1Y2Npb25fZGV2b2x1Y2lvbmVzPTAuMDUsIGNvc3RlX3ZvY19ldml0YWRvPTFfMDAwLjApLAogICAgImJhc2UiOiAgICAgICAgU3VwdWVzdG9zKCksCiAgICAib3B0aW1pc3RhIjogICBTdXB1ZXN0b3MocmVkdWNjaW9uX2Rldm9sdWNpb25lcz0wLjE1LCB1cGxpZnRfY29udmVyc2lvbj0wLjAwNSksCn0KCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgZm9yIG5vbWJyZSwgcyBpbiBFU0NFTkFSSU9TLml0ZW1zKCk6CiAgICAgICAgciA9IGNhbGN1bGFyKHMpCiAgICAgICAgcHJpbnQoZiJbe25vbWJyZTo+MTF9XSB2YWxvciB7clsndmFsb3JfdG90YWwnXTo+OSx9IEVVUi9hbmlvIHwgIgogICAgICAgICAgICAgIGYiUk9JIHtyWydyb2lfeCddOj42fXggfCBwYXliYWNrIHtyWydwYXliYWNrX2RpYXMnXX0gZGlhcyIpCg=='
open(BASE+'/resultados/caso_negocio_roi.md','wb').write(base64.b64decode(B64_MD))
open(BASE+'/opyna_roi.py','wb').write(base64.b64decode(B64_PY))
spec=importlib.util.spec_from_file_location('opyna_roi', BASE+'/opyna_roi.py')
mod=importlib.util.module_from_spec(spec); sys.modules['opyna_roi']=mod; spec.loader.exec_module(mod)
res={n: mod.calcular(s) for n,s in mod.ESCENARIOS.items()}
json.dump(res, open(BASE+'/resultados/roi_escenarios.json','w'), ensure_ascii=False, indent=2)
for n,r in res.items(): print(n, '-> valor', r['valor_total'], '| ROI', r['roi_x'], '| payback_dias', r['payback_dias'])
mem=BASE+'/memoria_resultados_200k.md'
if '## 13. Caso de negocio' not in open(mem,encoding='utf-8').read():
    L=['','## 13. Caso de negocio / ROI (8 de la propuesta)','- Modelo parametrizable (opyna_roi.py). Escenario base: valor ~104.000 EUR/anio, coste ~1.000 EUR, ROI ~100x.','- Palanca robusta: ahorro de analisis (VoC evitado). Palanca de impacto: reduccion de devoluciones (soporte/durabilidad).','- Entregables: resultados/caso_negocio_roi.md, resultados/roi_escenarios.json, opyna_roi.py.']
    open(mem,'a').write(chr(10).join(L)+chr(10)); print('memoria: seccion 13 anadida')
else: print('memoria: seccion 13 ya existia')
print('OK Paso12 ROI -> Drive')